# When do Daily Top Episodes start seeing your agent?

Kaggriculture publishes a daily sample of high-rated matches. I used 28,201 unique
episodes from 53 downloaded manifests (2026-07-30 through
2026-09-07) to estimate when a rising submission begins to enter that sample.

## Bottom line

**My operational trigger is rating 2,700.** At that point I would start fielding the two
newer low-rated submissions needed to move the strong submission out of Kaggle's latest-two
tracking set. Waiting for 2,800 is too late: a stronger opponent can lift the match-average
rating into the daily selection band before the submission itself reaches 2,800.

At the 2026-09-08 12:58:57 UTC snapshot, my current submission was already at **2,709.3**,
so it had reached this trigger.

Removing a submission from the daily sample does not make its episodes inaccessible. Public
Episode API calls can still retrieve a submission's individual public episode history.

## Data and calculation

The official manifest provides `avg_score`, `min_score`, and `sum_score` for every two-player
match. I reconstruct the higher player's rating as `sum_score - min_score`, and deduplicate
repeated archive imports by `episode_id`.

For the exposure trigger I use:

`trigger = lowest recent daily p05 match-average rating - (my p90 matchmaking gap / 2)`

The last seven daily p05 boundaries ranged from 2,767 to 2,894. My submission's p90 absolute
matchmaking gap was 108 points, so a stronger opponent can add about 54 points to the match
average. The conservative boundary is therefore `2,767 - 108/2 = 2,713`, rounded down to
**2,700** to leave execution time for two replacement submissions.

In [1]:
from io import StringIO

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.1f}")

DAILY_CSV = 'date,matches,avg_min,avg_p05,avg_p10,avg_p25,avg_median,avg_p75,avg_p90,lower_player_median,higher_player_median,gap_median,gap_p90\n2026-07-30,864,254.185774,396.667652,464.510714,561.535605,670.010652,811.561211,934.098840,616.102272,731.231293,114.061203,189.959612\n2026-07-31,928,1106.908203,1112.163074,1118.418041,1138.883554,1175.605931,1232.896666,1294.746634,1148.981666,1202.480081,51.952248,106.602568\n2026-08-01,829,1301.693778,1306.602621,1310.497844,1323.451434,1348.186244,1387.865193,1450.838543,1327.327727,1372.241759,44.333210,106.347215\n2026-08-02,793,2160.682753,2174.461399,2186.157898,2233.707145,2319.161237,2403.857919,2467.890282,2292.681236,2353.471872,51.910927,139.867148\n2026-08-03,787,2659.543163,2663.931525,2670.004848,2689.714563,2730.385790,2775.994116,2832.216389,2708.027720,2753.753736,44.151329,103.126945\n2026-08-04,755,2694.329358,2699.888436,2706.382182,2725.563695,2767.121715,2832.600271,2883.427749,2745.916849,2789.809230,40.091395,100.050682\n2026-08-05,743,2783.938565,2787.175606,2792.020841,2807.594465,2838.605560,2878.059691,2912.046692,2818.810876,2862.448481,37.100808,84.492750\n2026-08-06,683,2940.852554,2943.195478,2945.919885,2955.397576,2973.372293,2994.992368,3016.133641,2951.710318,2998.576142,38.741404,87.059904\n2026-08-07,675,3004.830303,3006.281898,3008.846770,3015.947019,3028.590465,3054.169396,3085.330245,3013.947847,3047.056625,28.638062,77.805712\n2026-08-08,675,3033.482287,3036.018415,3040.067194,3049.107832,3067.745951,3094.576692,3123.242395,3045.449028,3094.178468,42.964660,86.500739\n2026-08-09,687,3039.915280,3041.789043,3044.349244,3052.506293,3068.009365,3095.195421,3134.830753,3052.853536,3088.981095,35.265755,82.419036\n2026-08-10,685,3047.674346,3049.990714,3053.089534,3060.859177,3079.658269,3104.305674,3142.584173,3061.649081,3099.686931,34.017148,76.781555\n2026-08-11,682,3036.046529,3038.246084,3040.308798,3048.959233,3066.775297,3095.413193,3142.262212,3048.557380,3087.075199,35.932829,83.405645\n2026-08-12,684,3039.769039,3041.517494,3043.158488,3049.939389,3066.477792,3094.946060,3127.964117,3048.992900,3086.402124,34.315778,90.095301\n2026-08-13,687,3006.282324,3009.079941,3011.494769,3020.018558,3038.294970,3071.375378,3124.484644,3018.018132,3062.410544,36.171126,96.040947\n2026-08-14,690,2996.354968,3000.670803,3005.111477,3018.682337,3045.149114,3075.164095,3114.229558,3026.798868,3065.673415,37.919537,85.335235\n2026-08-15,697,2952.013319,2955.252690,2960.549085,2976.211666,3010.130894,3062.077140,3121.487241,2989.175167,3032.269777,43.183861,113.544561\n2026-08-16,699,2920.458030,2923.744245,2927.373986,2939.136612,2964.450916,3013.735018,3110.288115,2947.279611,2982.252018,40.128538,117.619714\n2026-08-17,699,2874.461869,2879.977952,2886.001790,2902.858448,2929.397458,2966.758751,3022.325372,2910.367575,2953.006133,36.973970,89.531234\n2026-08-18,697,2808.362729,2811.438728,2815.454926,2829.943796,2862.585582,2914.198340,2962.391457,2843.327607,2885.693596,35.400461,85.620223\n2026-08-19,695,2835.718108,2841.558677,2845.751324,2862.857396,2892.949144,2924.037877,2995.360497,2874.558140,2910.651634,37.587140,89.898901\n2026-08-20,698,2768.268686,2772.620179,2777.267789,2793.410947,2832.050253,2904.221327,3015.590409,2810.315813,2852.576012,36.972311,90.537528\n2026-08-21,697,2743.684744,2750.360065,2756.400836,2777.124580,2818.282167,2906.936568,3009.554888,2801.679968,2838.117606,40.297180,92.232061\n2026-08-22,695,2725.401312,2728.812189,2734.614567,2757.509738,2803.878576,2908.112050,3009.323554,2782.701256,2829.782059,38.325126,97.001071\n2026-08-23,692,2727.877754,2734.086832,2740.955157,2758.182650,2791.798005,2871.930999,2940.013579,2773.043574,2811.534896,37.729143,85.621278\n2026-08-24,692,2751.674856,2755.845964,2762.313860,2779.646716,2810.352537,2862.929798,2928.855382,2792.749099,2828.435718,39.269728,82.476974\n2026-08-25,688,2703.401417,2707.710412,2711.188608,2723.861180,2761.666096,2838.123809,2899.979497,2739.808324,2787.055684,33.467118,84.826295\n2026-08-26,687,2675.995811,2682.900184,2691.385993,2709.600952,2752.948115,2827.216905,2963.742438,2730.614322,2770.154038,35.391583,87.147227\n2026-08-27,696,2694.681053,2702.779817,2712.709601,2735.027190,2774.061433,2828.686880,2913.894467,2750.274358,2796.047482,41.470049,95.075160\n2026-08-28,676,2702.391300,2707.687791,2712.994040,2731.804187,2770.408155,2819.516084,2923.063903,2754.079741,2786.116639,36.881881,83.348561\n2026-08-29,667,2721.635225,2726.871862,2732.835512,2755.537410,2797.223938,2847.978025,2918.406571,2776.421666,2816.425528,36.564830,88.072216\n2026-08-30,664,2711.267225,2714.747927,2719.217870,2734.091238,2767.479293,2806.175474,2869.123348,2745.467532,2788.592495,35.444427,88.095806\n2026-08-31,665,2687.173798,2690.096416,2694.390742,2709.583511,2734.668997,2776.639450,2822.878013,2715.494916,2757.262337,34.777343,90.151103\n2026-09-01,664,2763.487806,2767.221459,2770.496251,2782.233635,2802.825164,2838.925876,2873.531993,2783.888141,2828.025196,39.225131,83.994186\n2026-09-02,660,2856.998888,2858.755488,2860.924557,2867.090844,2883.560765,2905.854000,2922.096139,2865.662492,2905.265972,28.009326,81.411271\n2026-09-03,669,2891.784772,2893.814863,2895.354174,2902.018242,2912.860207,2929.422853,2950.789746,2896.193285,2934.789444,35.447355,79.503393\n2026-09-04,668,2881.988340,2885.868717,2889.699095,2900.359707,2930.348294,2966.190940,2997.082156,2904.351825,2961.633586,40.060496,97.933898\n2026-09-05,664,2845.250732,2847.538921,2849.949189,2858.879769,2877.606318,2912.731435,2992.702646,2859.320978,2897.675814,42.155348,98.062539\n2026-09-06,662,2814.310316,2817.360948,2820.402488,2836.274494,2862.746740,2905.017090,2947.952183,2836.474239,2893.786602,49.378264,105.162501\n2026-09-07,663,2800.465739,2802.783372,2805.355277,2815.328593,2834.194606,2860.776230,2895.247054,2812.619697,2858.281178,42.106553,97.915821\n'
daily = pd.read_csv(StringIO(DAILY_CSV), parse_dates=["date"]).set_index("date")
CURRENT_RATING = 2709.250382670357
OWN_GAP_P90 = 108.06921868978154
OBSERVED_FLOOR = np.float64(2767.22145895)
TRIGGER_EXACT = np.float64(2713.1868496051093)
SNAPSHOT_DATE = pd.Timestamp("2026-09-08")
daily.tail(7)

## How the recorded rating band moved

The solid line is the daily median match-average rating. The dark band is the central 50%,
and the light band is p10–p90. The early competition ramp is large; the last two weeks are
shown separately below so recent increases and decreases remain visible.

In [2]:
daily["median_change"] = daily["avg_median"].diff()
fig, axes = plt.subplots(2, 1, figsize=(13, 8), constrained_layout=True)

x = daily.index
axes[0].fill_between(x, daily["avg_p10"], daily["avg_p90"], color="#60a5fa", alpha=0.18, label="p10–p90")
axes[0].fill_between(x, daily["avg_p25"], daily["avg_p75"], color="#2563eb", alpha=0.25, label="p25–p75")
axes[0].plot(x, daily["avg_median"], color="#1d4ed8", linewidth=2.5, marker="o", markersize=3, label="median")
axes[0].set(title="Distribution of match-average rating", ylabel="rating")
axes[0].legend(ncol=3)

recent = daily.tail(14)
colors = np.where(recent["median_change"].ge(0), "#16a34a", "#dc2626")
axes[1].axhline(0, color="#334155", linewidth=0.8)
axes[1].bar(recent.index, recent["median_change"], width=0.8, color=colors)
axes[1].set(title="Day-over-day median change: latest 14 days", ylabel="rating points", xlabel="UTC date")
for ax in axes:
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=6, maxticks=12))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.show()

## The exposure frontier

The blue line is the low edge of the daily sample (its p05 match-average rating). The orange
line subtracts half of a 108-point opponent gap and is the rating at which a submission can
enter the sample when matched upward. The dashed line is the rounded operational trigger.

In [3]:
daily["exposure_frontier"] = daily["avg_p05"] - OWN_GAP_P90 / 2
recent = daily.tail(14)

fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
ax.plot(recent.index, recent["avg_p05"], color="#2563eb", marker="o", label="daily p05 match average")
ax.plot(recent.index, recent["exposure_frontier"], color="#ea580c", marker="o", label="p05 minus half p90 gap")
ax.axhline(2700, color="#111827", linestyle="--", linewidth=1.5, label="operational trigger: 2,700")
ax.scatter([SNAPSHOT_DATE], [CURRENT_RATING], color="#7c3aed", s=80, zorder=5, label=f"my 09-08 snapshot: {CURRENT_RATING:,.1f}")
ax.set(title="Estimated entry point into Daily Top Episodes", ylabel="rating", xlabel="UTC date")
ax.legend(ncol=2)
ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=7, maxticks=14))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
plt.show()

display(daily.tail(7)[["matches", "avg_p05", "avg_median", "gap_p90", "exposure_frontier"]])

## Operational summary

1. **Begin at 2,700**, not 2,800.
2. Submit **two newer low-rated agents**; one newer submission still leaves the strong one in
   the latest-two tracked set.
3. Use the rolling formula rather than treating 2,700 as permanent. Recompute it when the next
   daily manifest arrives.
4. This reduces inclusion in the bulk daily archive. It does not block submission-specific
   retrieval through the public Episode API.